## 클로드는 요청을 어떻게 실제 작업으로 옮기는가

여러분이 지금까지 실습하면서 클로드에게 내린 명령들, 클로드는 대체 어떻게 작업을 수행하는 걸까요?

대체로 컴퓨터 내에서 작업은 터미널 명령어와 파이썬 코드를 활용해 수행합니다. 여러분이 질문을 했을 때, 클로드가 어떤 반응을 보이는지 대략적으로 확인해 볼까요?

---
## 상황 설정

프로젝트를 진행하다 보니 폴더가 지저분해졌습니다. 지점별 매출 파일도 있고, 작업하다 생긴 임시 파일도 섞여 있습니다.

아래는 사용자가 클로드에게 요청을 했을 때, 클로드가 **무엇을 생각하고 → 어떤 명령을 만들고 → 그 결과를 보고 다음에 무엇을 하는지**를 순서대로 따라가 본 기록입니다.

---
### 🙋 사용자

> **"현재 폴더에 있는 파일들 확인하고, 쓸모 없는 애들은 정리해줘."**

### 🤖 클로드

네, 알겠습니다. 삭제는 되돌리기 어려운 작업이라 바로 지우지 않고 단계를 나눠서 진행하겠습니다.

우선 **현재 폴더에 어떤 파일이 있는지부터** 확인해 볼게요.

In [ ]:
!dir

### 🤖 클로드

목록을 보니 이 폴더는 **프로젝트를 수행하면서 생성된 파일들**이 섞여 있는 것으로 보입니다. 크게 세 종류로 나뉩니다.

- `강남지점_매출.xlsx`, `부산지점_매출.xlsx`, `인천지점_매출.xlsx` → 실제 업무 데이터
- `회의자료_초안.pptx` → 아직 쓰고 있는 문서로 보임
- `temp_export.tmp`, `backup_old.tmp`, `debug_0825.log` → 이름과 확장자로 보아 작업 중 생긴 부산물

다만 **이름만 보고 바로 지우는 건 위험**합니다. 파일 크기와 마지막 수정 날짜까지 같이 확인해서, 정말 오래되고 안 쓰는 파일인지 근거를 만들어 보겠습니다. 이 정도 판단은 명령어 한 줄로는 어려워서 파이썬으로 처리하겠습니다.

In [ ]:
import os
from datetime import datetime

# 작업 중 자동으로 생기는 부산물로 볼 확장자
junk_exts = [".tmp", ".log", ".bak"]

print(f"{'파일명':<28}{'크기(KB)':>10}{'수정일':>14}   판정")
print("-" * 72)

for filename in os.listdir():
    if os.path.isdir(filename):
        continue
    ext = os.path.splitext(filename)[1].lower()
    size_kb = os.path.getsize(filename) / 1024
    modified = datetime.fromtimestamp(os.path.getmtime(filename)).strftime("%Y-%m-%d")
    verdict = "[정리 후보]" if ext in junk_exts else "[보존]"
    print(f"{filename:<28}{size_kb:>10.1f}{modified:>14}   {verdict}")

### 🤖 클로드

정리 후보는 아래 3개입니다.

- `temp_export.tmp`
- `backup_old.tmp`
- `debug_0825.log`

엑셀 매출 파일과 회의자료 초안은 실제 업무 파일이라 손대지 않겠습니다.

지금 바로 삭제해도 될까요? **되돌릴 수 없으니** 확인 후 진행하겠습니다.

### 🙋 사용자

> **"응, 그 3개는 지워도 돼."**

### 🤖 클로드

알겠습니다. 삭제를 진행하겠습니다.

In [ ]:
!del *.tmp
!del *.log

### 🤖 클로드

삭제했습니다. 의도한 파일만 지워졌는지, 남아 있어야 할 파일이 그대로 있는지 **다시 확인**해 보겠습니다.

In [ ]:
!dir

### 🤖 클로드

정리 완료되었습니다.

- 삭제: `temp_export.tmp`, `backup_old.tmp`, `debug_0825.log` (3개)
- 보존: 지점별 매출 파일 3개, `회의자료_초안.pptx`

업무 파일은 그대로 남아 있습니다.

---
### 🙋 사용자

> **"고마워. 그럼 이제 남아 있는 지점별 매출 파일들, 품목별로 합계 좀 내줘."**

### 🤖 클로드

네. 이번 요청은 파일을 지우거나 옮기는 게 아니라 **파일 안의 숫자를 읽어서 계산**해야 합니다.

계산에 들어가기 전에, 세 파일이 같은 형식인지부터 확인하겠습니다. 열 이름이 다르면 합칠 수가 없기 때문입니다.

In [ ]:
import pandas as pd

excel_files = ["강남지점_매출.xlsx", "부산지점_매출.xlsx", "인천지점_매출.xlsx"]

for filename in excel_files:
    df = pd.read_excel(filename)
    print(f"[{filename}]  행 {len(df)}개")
    print(f"  열 구성: {list(df.columns)}")
    print()

### 🤖 클로드

세 파일 모두 **동일한 열 구성**을 가지고 있습니다. 그대로 하나로 합쳐도 문제 없겠습니다.

다만 합치고 나면 어느 지점 데이터인지 구분이 안 되므로, **파일명에서 지점 이름을 따와 열로 추가**하면서 합치겠습니다.

In [ ]:
tables = []

for filename in excel_files:
    df = pd.read_excel(filename)
    df["지점"] = filename.replace("_매출.xlsx", "")   # 파일명에서 지점명 추출
    tables.append(df)

all_sales = pd.concat(tables, ignore_index=True)

print(f"합친 결과: 총 {len(all_sales)}행")
all_sales.head(10)

### 🤖 클로드

세 지점 데이터가 하나의 표로 합쳐졌습니다. 이제 요청하신 **품목별 매출 합계**를 계산하겠습니다.

In [ ]:
summary = (
    all_sales.groupby("품목")["매출액(원)"]
    .sum()
    .reset_index()
    .sort_values("매출액(원)", ascending=False)
)

summary

### 🤖 클로드

계산이 끝났습니다. 화면으로만 보면 다시 쓰기 불편하니, **엑셀 파일로 저장**해 두겠습니다.

In [ ]:
summary.to_excel("품목별_매출합계.xlsx", index=False)
print("저장 완료 → 품목별_매출합계.xlsx")

### 🤖 클로드

작업을 마쳤습니다.

1. 폴더를 확인하고, 정리 후보를 근거와 함께 추려서 임시 파일 3개를 삭제했습니다.
2. 지점별 매출 파일 3개를 하나로 합쳐 품목별 매출 합계를 계산하고, `품목별_매출합계.xlsx`로 저장했습니다.

---
## 정리

지금 본 흐름에서 클로드가 반복한 패턴은 이렇습니다.

| 단계 | 클로드가 한 일 |
|---|---|
| ① 파악 | 바로 실행하지 않고, 먼저 **현재 상태를 확인**하는 명령을 실행 |
| ② 판단 | 결과를 읽고 **무엇을 해야 하는지 근거를 정리** |
| ③ 확인 | 되돌릴 수 없는 작업(삭제 등)은 **사용자에게 물어봄** |
| ④ 실행 | 요청에 맞는 명령·코드를 **직접 작성해서 실행** |
| ⑤ 검증 | 실행 후 **결과가 의도대로인지 다시 확인** |

또 하나, 요청의 성격에 따라 도구를 바꿔 썼습니다.

- 파일을 **찾고, 지우고, 옮기는** 요청 → 터미널 명령
- 파일 **안의 데이터를 읽고 계산·가공**하는 요청 → 파이썬 코드

즉 클로드는 "시키는 대로 한 줄 실행"하는 것이 아니라, **확인 → 판단 → 실행 → 검증**을 스스로 돌면서 일합니다. 다음 실습에서는 이 과정을 직접 시켜보게 됩니다.